# 01 — Подготовка данных

**Этап 1 задания на ВКР:** подготовка обучающих данных и базлайн.

Шаги:

0. Пути, импорты, токенизатор
1. Загрузка OpenR1-Math-220k (split default)
2. Разведка поля `source`, фильтрация
3. Выбор одного корректного решения на задачу
4. Токенизация (длины решений в токенах Qwen2.5-1.5B)
5. Загрузка тестовых наборов (GSM8K test, MATH-500)
6. Дедупликация (13-gram containment)
7. Отделение валидационной выборки
8. Формирование 12 подмножеств (4 объёма × 3 порога CoT)
9. Формат для TRL и сохранение
10. Статистика D_train
11. Базлайн (Qwen2.5-1.5B без дообучения)
12. Итоговая проверка

## Шаг 0. Пути, импорты, токенизатор

In [ ]:
!pip install -q datasets transformers pyyaml

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from google.colab import drive
import os
import json
import sys

drive.mount('/content/drive')

# Пути — те же, что в 00_setup
MODEL_ROOT = '/content/drive/MyDrive/vkr_models'
DATA_ROOT = '/content/drive/MyDrive/vkr_data'
RESULTS_ROOT = '/content/drive/MyDrive/vkr_results'

# Подпапка для обучающих подмножеств
SUBSETS_DIR = f'{DATA_ROOT}/subsets'
os.makedirs(SUBSETS_DIR, exist_ok=True)

# parsing.py из 00_setup
sys.path.insert(0, DATA_ROOT)
import parsing

print(f'Модели:       {MODEL_ROOT}')
print(f'Данные:       {DATA_ROOT}')
print(f'Подмножества: {SUBSETS_DIR}')
print(f'Результаты:   {RESULTS_ROOT}')
print(f'parsing.py:   {parsing.__file__}')

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_PATH = f'{MODEL_ROOT}/Qwen2.5-1.5B'
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# Проверка
test_text = '<think>\nLet me solve this step by step.\n</think>\n\nThe answer is \\boxed{42}.'
tokens = tokenizer.encode(test_text)
print(f'Токенизатор: {TOKENIZER_PATH}')
print(f'Размер словаря: {tokenizer.vocab_size}')
print(f'Тестовая строка: {len(test_text)} символов → {len(tokens)} токенов')
print(f'EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})')

## Шаг 1. Загрузка OpenR1-Math-220k

In [ ]:
from datasets import load_dataset

ds_raw = load_dataset('open-r1/OpenR1-Math-220k', split='train', name='default')

print(f'Загружено: {len(ds_raw)} задач')
print(f'Колонки: {ds_raw.column_names}')

In [ ]:
# Один пример — посмотреть структуру
ex = ds_raw[0]
print(f'problem:        {ex["problem"][:200]}...')
print(f'answer:         {ex["answer"]}')
print(f'source:         {ex["source"]}')
print(f'problem_type:   {ex["problem_type"]}')
print(f'question_type:  {ex["question_type"]}')
print(f'generations:    {len(ex["generations"])} шт, первое: {len(ex["generations"][0])} символов')
print(f'correctness:    {ex["correctness_math_verify"]}')
print(f'correct_count:  {ex["correctness_count"]}')

In [ ]:
# Первые 500 символов первого решения — посмотреть формат (теги <think>)
gen = ex['generations'][0]
print('--- Начало решения ---')
print(gen[:500])
print('...')
print('--- Конец решения ---')
print(gen[-200:])

## Шаг 2. Разведка поля `source`, фильтрация

Обучающая выборка — задачи школьного уровня.  
Олимпиадные исключаем: MATH-500 содержит олимпиадные задачи,  
и мы хотим проверять обобщение, а не запоминание.

In [ ]:
from collections import Counter

source_counts = Counter(ds_raw['source'])

print(f'Уникальных значений source: {len(source_counts)}')
print()
print(f'{"source":<25s} {"count":>7s} {"доля":>7s}')
print('-' * 42)
for src, cnt in source_counts.most_common():
    print(f'{src:<25s} {cnt:>7d} {cnt/len(ds_raw):>7.1%}')

В ходе работы установлено, что default split датасета OpenR1-Math-220k содержит задачи конкурсного, а не школьного уровня; формулировки скорректированы.

## Шаг 3. Выбор одного корректного решения на задачу

Каждая задача имеет 2–4 генерации от DeepSeek-R1.  
Берём первое решение с `correctness_math_verify == True`.  
Задачи без корректных решений отбрасываем (не должно быть в default split).

In [ ]:
def extract_solution(example):
    gens = example['generations']
    math_flags = example.get('correctness_math_verify') or []
    llama_flags = example.get('correctness_llama') or []

    for i, gen in enumerate(gens):
        if i < len(math_flags) and math_flags[i]:
            return {'solution': gen, 'solution_source': 'math_verify'}

    for i, gen in enumerate(gens):
        if i < len(llama_flags) and llama_flags[i]:
            return {'solution': gen, 'solution_source': 'llama'}

    return {'solution': '', 'solution_source': 'none'}

# Добавляем колонку solution, не дублируя весь датасет
ds_clean = ds_raw.map(extract_solution)
ds_clean = ds_clean.filter(lambda x: x['solution'] != '')
ds_clean = ds_clean.select_columns(['problem', 'answer', 'source', 'solution'])

# Освобождаем память
del ds_raw
import gc; gc.collect()

print(f'{len(ds_clean)} задач, колонки: {ds_clean.column_names}')

## Шаг 4. Токенизация

In [ ]:
ds_clean = ds_clean.map(
    lambda x: {'n_tokens': len(tokenizer.encode(x['solution']))},
    desc='Токенизация'
)

# Статистика
import numpy as np

lengths = np.array(ds_clean['n_tokens'])
print(f'Задач: {len(lengths)}')
print(f'Min: {lengths.min()}, Max: {lengths.max()}, Среднее: {lengths.mean():.0f}, Медиана: {np.median(lengths):.0f}')
print()
for p in [25, 50, 75, 90, 95, 99]:
    print(f'  P{p}: {np.percentile(lengths, p):.0f}')

# Сколько задач пройдёт каждый порог
print()
for threshold in [2048, 4096, 8192]:
    count = int((lengths <= threshold).sum())
    print(f'  ≤ {threshold}: {count} задач ({count/len(lengths):.1%})')

## Шаг 5. Загрузка тестовых наборов

In [ ]:
from datasets import load_dataset

# GSM8K test
ds_gsm8k = load_dataset('openai/gsm8k', 'main', split='test')
print(f'GSM8K test: {len(ds_gsm8k)} задач')
assert len(ds_gsm8k) == 1319, f'Ожидалось 1319, получено {len(ds_gsm8k)}'

# MATH-500 (подмножество MATH, Lightman et al.)
ds_math500 = load_dataset('HuggingFaceH4/MATH-500', split='test')
print(f'MATH-500:   {len(ds_math500)} задач')
assert len(ds_math500) == 500, f'Ожидалось 500, получено {len(ds_math500)}'

print('n/GSM8K пример:')
print(f'  question: {ds_gsm8k[0]["question"][:100]}...')
print(f'  answer:   {ds_gsm8k[0]["answer"][-50:]}')
print('n/MATH-500 пример:')
print(f'  problem:  {ds_math500[0]["problem"][:100]}...')
print(f'  answer:   {ds_math500[0]["answer"]}')

## Шаг 6. Дедупликация (13-gram containment)

In [ ]:
import re
from collections import defaultdict

def normalize(text):
    text = text.lower()
    text = re.sub(r'\\[a-z]+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def get_ngrams(text, n=13):
    words = normalize(text).split()
    if len(words) < n:
        return set()
    return {tuple(words[i:i+n]) for i in range(len(words) - n + 1)}

# Индекс тестовых задач
test_problems = [(s, t) for s, t in
    [('gsm8k', ex['question']) for ex in ds_gsm8k] +
    [('math500', ex['problem']) for ex in ds_math500]]

inv_index = defaultdict(set)
test_ngrams_all = {}  # сохраняем сами множества для верификации

for tid, (_, text) in enumerate(test_problems):
    ngs = get_ngrams(text)
    test_ngrams_all[tid] = ngs
    for ng in ngs:
        inv_index[ng].add(tid)

print(f'Тестовых задач: {len(test_problems)}, уникальных 13-грамм: {len(inv_index)}')

In [ ]:
# Поиск с верификацией каждого совпадения
contaminated = set()
log = []

for idx in range(len(ds_clean)):
    train_ngs = get_ngrams(ds_clean[idx]['problem'])
    if not train_ngs:
        continue

    hits = defaultdict(int)
    for ng in train_ngs:
        for tid in inv_index.get(ng, ()):
            hits[tid] += 1

    for tid, count in hits.items():
        if len(test_ngrams_all[tid]) == 0:
            continue
        containment = count / len(test_ngrams_all[tid])
        if containment >= 0.5:
            # Верификация: пересчитываем пересечение напрямую
            real_overlap = len(train_ngs & test_ngrams_all[tid])
            real_containment = real_overlap / len(test_ngrams_all[tid])
            if real_containment >= 0.5:
                contaminated.add(idx)
                log.append({
                    'train_idx': idx,
                    'test_source': test_problems[tid][0],
                    'containment': round(real_containment, 3),
                    'train': ds_clean[idx]['problem'][:300],
                    'test': test_problems[tid][1][:300]
                })
                break
            else:
                print(f'  ЛОЖНОЕ СРАБАТЫВАНИЕ: idx={idx}, tid={tid}')
                print(f'    inv_index containment={containment:.3f}, real={real_containment:.3f}')

    if idx % 20000 == 0:
        print(f'  {idx}/{len(ds_clean)}...')

print(f'\nУтечек: {len(contaminated)}')

with open(f'{RESULTS_ROOT}/dedup_log.json', 'w') as f:
    json.dump(log, f, indent=2, ensure_ascii=False)

if contaminated:
    keep = sorted(set(range(len(ds_clean))) - contaminated)
    ds_clean = ds_clean.select(keep)

print(f'После дедупликации: {len(ds_clean)}')

In [ ]:
for entry in log:
    print(f'{entry["test_source"]}, containment={entry["containment"]}')
    print(f'  train: {entry["train"][:120]}')
    print(f'  test:  {entry["test"][:120]}')
    print()

## Шаг 7. Отделение валидационной выборки

In [ ]:
VAL_SIZE = 500
VAL_SEED = 0

# Перемешиваем и отделяем
ds_shuffled = ds_clean.shuffle(seed=VAL_SEED)
ds_val = ds_shuffled.select(range(VAL_SIZE))
ds_pool = ds_shuffled.select(range(VAL_SIZE, len(ds_shuffled)))

print(f'Val:  {len(ds_val)}')
print(f'Pool: {len(ds_pool)}')

# Проверка: val ∩ pool = ∅
val_problems = set(ds_val['problem'])
pool_problems = set(ds_pool['problem'])
assert len(val_problems & pool_problems) == 0, 'Val и pool пересекаются!'
print('Val ∩ Pool = ∅ ✓')

In [ ]:
from collections import Counter

pool_counts = Counter(ds_pool['source'])
val_counts = Counter(ds_val['source'])
all_sources = sorted(set(list(pool_counts.keys()) + list(val_counts.keys())), key=lambda s: -pool_counts[s])

print(f'{"source":<20s} {"pool":>7s} {"pool%":>7s} {"val":>5s} {"val%":>7s}')
print('-' * 50)
for src in all_sources:
    pc = pool_counts[src]
    vc = val_counts.get(src, 0)
    print(f'{src:<20s} {pc:>7d} {pc/len(ds_pool):>7.1%} {vc:>5d} {vc/len(ds_val):>7.1%}')
print('-' * 50)
print(f'{"ИТОГО":<20s} {len(ds_pool):>7d} {"100%":>7s} {len(ds_val):>5d} {"100%":>7s}')

## Шаг 8. Формирование 12 подмножеств (4 объёма × 3 порога CoT)

In [ ]:
THRESHOLDS = [2048, 4096, 8192]
VOLUMES = [500, 1000, 2000, 5000]
SAMPLE_SEED = 42

subsets = {}

for threshold in THRESHOLDS:
    # Фильтр по длине CoT
    pool_filtered = ds_pool.filter(lambda x: x['n_tokens'] <= threshold)
    print(f'Порог {threshold}: {len(pool_filtered)} задач в подпуле')
    assert len(pool_filtered) >= 5000, f'Недостаточно задач для порога {threshold}!'

    # Перемешиваем, берём вложенные подмножества
    pool_shuffled = pool_filtered.shuffle(seed=SAMPLE_SEED)
    for vol in VOLUMES:
        key = f'c{threshold}_v{vol}'
        subsets[key] = pool_shuffled.select(range(vol))
        print(f'  {key}: {len(subsets[key])} задач')

print(f'\nВсего подмножеств: {len(subsets)}')

In [ ]:
# Проверка вложенности: v500 ⊂ v1000 ⊂ v2000 ⊂ v5000

for threshold in THRESHOLDS:
    for i in range(len(VOLUMES) - 1):
        small = set(subsets[f'c{threshold}_v{VOLUMES[i]}']['problem'])
        big = set(subsets[f'c{threshold}_v{VOLUMES[i+1]}']['problem'])
        assert small.issubset(big), f'Вложенность нарушена: c{threshold}_v{VOLUMES[i]} ⊄ c{threshold}_v{VOLUMES[i+1]}'
    print(f'c{threshold}: v500 ⊂ v1000 ⊂ v2000 ⊂ v5000 ✓')

# Проверка: подмножества ∩ val = ∅
val_problems = set(ds_val['problem'])
for key, ds in subsets.items():
    overlap = val_problems & set(ds['problem'])
    assert len(overlap) == 0, f'{key} пересекается с val!'
print('Все подмножества ∩ val = ∅ ✓')

## Шаг 9. Формат для TRL и сохранение.

Каждое подмножество конвертируется в формат messages и сохраняется на Drive в SUBSETS_DIR. Плюс val. Итого 13 файлов.

In [ ]:
def to_messages(example):
    return {'messages': [
        {'role': 'user', 'content': example['problem'] + '\n\nPlease reason step by step, and put your final answer within \\boxed{}.'},
        {'role': 'assistant', 'content': example['solution']}
    ]}

# Сохраняем 12 подмножеств
for key, ds in subsets.items():
    ds_msg = ds.map(to_messages, remove_columns=ds.column_names)
    path = f'{SUBSETS_DIR}/{key}'
    ds_msg.save_to_disk(path)
    print(f'{key}: {len(ds_msg)} → {path}')

# Сохраняем val
ds_val_msg = ds_val.map(to_messages, remove_columns=ds_val.column_names)
ds_val_msg.save_to_disk(f'{SUBSETS_DIR}/val')
print(f'val: {len(ds_val_msg)} → {SUBSETS_DIR}/val')

print(f'\nСохранено: {len(subsets) + 1} датасетов')

In [ ]:
# Проверка: загружаем один датасет и смотрим формат
from datasets import load_from_disk

check = load_from_disk(f'{SUBSETS_DIR}/c2048_v500')
print(f'Загружено: {len(check)} примеров')
print(f'Колонки: {check.column_names}')
print()
ex = check[0]
print(f'user:      {ex["messages"][0]["content"][:150]}...')
print(f'assistant: {ex["messages"][1]["content"][:150]}...')

In [ ]:
from datasets import load_from_disk
import os

# 1. Все 13 датасетов загружаются и имеют правильный размер
expected = {f'c{t}_v{v}': v for t in [2048, 4096, 8192] for v in [500, 1000, 2000, 5000]}
expected['val'] = 500

errors = []
for name, size in expected.items():
    path = f'{SUBSETS_DIR}/{name}'
    if not os.path.exists(path):
        errors.append(f'{name}: не найден')
        continue
    ds = load_from_disk(path)
    if len(ds) != size:
        errors.append(f'{name}: ожидалось {size}, получено {len(ds)}')
    if ds.column_names != ['messages']:
        errors.append(f'{name}: колонки {ds.column_names}, ожидалось [messages]')
    # Проверка формата первого примера
    ex = ds[0]['messages']
    if len(ex) != 2:
        errors.append(f'{name}: messages длина {len(ex)}, ожидалось 2')
    elif ex[0]['role'] != 'user' or ex[1]['role'] != 'assistant':
        errors.append(f'{name}: роли {ex[0]["role"]}, {ex[1]["role"]}')
    elif '\\boxed{}' not in ex[0]['content']:
        errors.append(f'{name}: нет инструкции в user prompt')
    elif '<think>' not in ex[1]['content']:
        errors.append(f'{name}: нет <think> в assistant')

if errors:
    print('ОШИБКИ:')
    for e in errors:
        print(f'  {e}')
else:
    print(f'Все {len(expected)} датасетов: размеры ✓, формат ✓, промпт ✓, <think> ✓')

## Шаг 10. Статистика D_train

Для каждого из 12 подмножеств считаем суммарное число токенов (prompt + solution с учётом chat template).  
D_train = total_tokens × n_epochs. C_SFT = 6ND. Результат — данные для Парето-фронта (задача 4).

In [ ]:
stats = {}

for threshold in [2048, 4096, 8192]:
    for vol in [500, 1000, 2000, 5000]:
        key = f'c{threshold}_v{vol}'
        ds = load_from_disk(f'{SUBSETS_DIR}/{key}')

        total_tokens = 0
        for ex in ds:
            text = tokenizer.apply_chat_template(ex['messages'], tokenize=False)
            total_tokens += len(tokenizer.encode(text))

        avg = total_tokens / len(ds)
        stats[key] = {'examples': len(ds), 'total_tokens': total_tokens, 'avg_tokens': round(avg)}
        print(f'{key}: {vol} примеров, {total_tokens:>10,} токенов, avg={avg:.0f}')

N_EPOCHS = 3
N = 1.5e9
print(f'\nD_train (×{N_EPOCHS} эпох):')
print(f'{"subset":<16s} {"tokens":>12s} {"D_train":>12s} {"C_SFT (6ND)":>15s}')
print('-' * 58)
for key, s in stats.items():
    d_train = s['total_tokens'] * N_EPOCHS
    c_sft = 6 * N * d_train
    print(f'{key:<16s} {s["total_tokens"]:>12,} {d_train:>12,} {c_sft:>15.2e}')

import json
with open(f'{RESULTS_ROOT}/data_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(f'\nСохранено: {RESULTS_ROOT}/data_stats.json')

In [ ]:
print(f'{"Подмножество":<16s} │ {"Примеров":>8s} │ {"Токенов":>12s} │ {"Avg":>6s} │ {"D_train (×3)":>12s} │ {"C_SFT (FLOPs)":>12s}')
print('─' * 80)

N = 1.5e9
N_EPOCHS = 3
prev_prefix = ''

for key, s in stats.items():
    prefix = key.split('_')[0]
    if prev_prefix and prefix != prev_prefix:
        print('─' * 80)
    prev_prefix = prefix

    d_train = s['total_tokens'] * N_EPOCHS
    c_sft = 6 * N * d_train
    print(f'{key:<16s} │ {s["examples"]:>8,} │ {s["total_tokens"]:>12,} │ {s["avg_tokens"]:>6,} │ {d_train:>12,} │ {c_sft:>12.2e}')

print('─' * 80)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

volumes = [500, 1000, 2000, 5000]
colors = {'c2048': '#2196F3', 'c4096': '#FF9800', 'c8192': '#4CAF50'}

# График 1: Средняя длина (avg токенов) по подмножествам
for prefix, color in colors.items():
    avgs = [stats[f'{prefix}_v{v}']['avg_tokens'] for v in volumes]
    ax1.plot(volumes, avgs, 'o-', color=color, label=f'CoT ≤ {prefix[1:]}', linewidth=2)

ax1.set_xlabel('Объём данных')
ax1.set_ylabel('Средняя длина (токены)')
ax1.set_title('Средняя длина примера')
ax1.legend()
ax1.grid(True, alpha=0.3)

# График 2: C_SFT (FLOPs) по подмножествам
N = 1.5e9
N_EPOCHS = 3
for prefix, color in colors.items():
    c_sft = [6 * N * stats[f'{prefix}_v{v}']['total_tokens'] * N_EPOCHS for v in volumes]
    ax2.plot(volumes, c_sft, 'o-', color=color, label=f'CoT ≤ {prefix[1:]}', linewidth=2)

ax2.set_xlabel('Объём данных')
ax2.set_ylabel('C_SFT (FLOPs)')
ax2.set_title('Расчётная стоимость SFT-дообучения')
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_ROOT}/data_stats_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Сохранено: {RESULTS_ROOT}/data_stats_plot.png')

## Шаг 11. Базлайн (Qwen2.5-1.5B без дообучения)

модель из коробки + промпт + тестовые задачи → смотрим, сколько решит.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model_path = f'{MODEL_ROOT}/Qwen2.5-1.5B'
model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16, device_map='auto')
model.eval()
device = next(model.parameters()).device

print(f'Модель: {model_path}')
print(f'Устройство: {device}')

In [ ]:
import re
import logging
import json
logging.getLogger('transformers.generation').setLevel(logging.ERROR)

from tqdm import tqdm

PROMPT = 'Please reason step by step, and put your final answer within \\boxed{}.'

stop_ids = [
    tokenizer.convert_tokens_to_ids('<|endoftext|>'),
    tokenizer.convert_tokens_to_ids('<|im_end|>')
]

def generate(problem):
    text = f'Problem: {problem}\n\nPlease reason step by step, and put your final answer within \\boxed{{}}.\n\nSolution:'
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False, eos_token_id=stop_ids)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def extract_answer_gsm8k_lenient(text):
    strict = parsing.extract_answer_gsm8k(text)
    if strict is not None:
        return strict
    numbers = re.findall(r'[-+]?\d[\d,]*\.?\d*', text)
    if numbers:
        return numbers[-1].replace(',', '').rstrip('.')
    return None


BASELINE_PARTIAL = f'{RESULTS_ROOT}/baseline_partial.json'

def eval_gsm8k():
    results = []
    for i, ex in enumerate(tqdm(ds_gsm8k, desc='GSM8K')):
        resp = generate(ex['question'])
        pred = extract_answer_gsm8k_lenient(resp)
        gold = parsing.extract_answer_gsm8k(ex['answer'])
        results.append({'correct': parsing.is_correct_gsm8k(pred, gold), 'pred': pred, 'none': pred is None})
        if i % 100 == 0:
            with open(BASELINE_PARTIAL, 'w') as f:
                json.dump({'gsm8k': results}, f)
    return results

def eval_math500():
    results = []
    for i, ex in enumerate(tqdm(ds_math500, desc='MATH-500')):
        resp = generate(ex['problem'])
        pred = parsing.extract_answer_math(resp)
        correct = parsing.is_equiv(pred, ex['answer']) if pred is not None else False
        results.append({'correct': correct, 'pred': pred, 'none': pred is None})
        if i % 100 == 0:
            with open(BASELINE_PARTIAL, 'w') as f:
                json.dump({'math500': results}, f)
    return results

In [ ]:
for i in range(5):
    resp = generate(ds_gsm8k[i]['question'])
    pred = extract_answer_gsm8k_lenient(resp)
    gold = parsing.extract_answer_gsm8k(ds_gsm8k[i]['answer'])
    print(f'Задача {i}: pred={pred}, gold={gold}, correct={parsing.is_correct_gsm8k(pred, gold)}')
    print(resp[:200])
    print('---')

In [ ]:
gsm8k_results = eval_gsm8k()
math_results = eval_math500()
print('Готово.')

In [ ]:
gsm8k_correct = [r['correct'] for r in gsm8k_results]
math_correct = [r['correct'] for r in math_results]

gsm8k_acc, gsm8k_lo, gsm8k_hi = parsing.bootstrap_ci(gsm8k_correct)
math_acc, math_lo, math_hi = parsing.bootstrap_ci(math_correct)

gsm8k_none = sum(r['none'] for r in gsm8k_results) / len(gsm8k_results)
math_none = sum(r['none'] for r in math_results) / len(math_results)

print(f'GSM8K:    acc={gsm8k_acc:.3f} [{gsm8k_lo:.3f}, {gsm8k_hi:.3f}]  none_rate={gsm8k_none:.1%}')
print(f'MATH-500: acc={math_acc:.3f} [{math_lo:.3f}, {math_hi:.3f}]  none_rate={math_none:.1%}')

if gsm8k_none > 0.05 or math_none > 0.05:
    print('\n⚠ none_rate > 5%!')

baseline = {
    'gsm8k': {'accuracy': gsm8k_acc, 'ci': [gsm8k_lo, gsm8k_hi], 'none_rate': gsm8k_none},
    'math500': {'accuracy': math_acc, 'ci': [math_lo, math_hi], 'none_rate': math_none}
}
with open(f'{RESULTS_ROOT}/baseline.json', 'w') as f:
    json.dump(baseline, f, indent=2)
print(f'Сохранено: {RESULTS_ROOT}/baseline.json')

# Освобождаем GPU
del model
torch.cuda.empty_cache()

## Шаг 12. Итоговая проверка


In [ ]:
## Шаг 12. Итоговая проверка

import os
from datasets import load_from_disk

checks = []

# Файлы на месте
for name in [f'c{t}_v{v}' for t in [2048, 4096, 8192] for v in [500, 1000, 2000, 5000]] + ['val']:
    checks.append((f'{name} существует', os.path.exists(f'{SUBSETS_DIR}/{name}')))

for path in ['dedup_log.json', 'baseline.json', 'data_stats.json']:
    checks.append((f'{path} существует', os.path.exists(f'{RESULTS_ROOT}/{path}')))

# Вложенность + непересечение с val
val_p = set(str(m) for m in load_from_disk(f'{SUBSETS_DIR}/val')['messages'])
vols = [500, 1000, 2000, 5000]

for t in [2048, 4096, 8192]:
    ds = {v: [str(m) for m in load_from_disk(f'{SUBSETS_DIR}/c{t}_v{v}')['messages']] for v in vols}
    for i in range(len(vols) - 1):
        checks.append((f'c{t}: v{vols[i]} ⊂ v{vols[i+1]}', set(ds[vols[i]]).issubset(ds[vols[i+1]])))
    checks.append((f'val ∩ c{t}_v5000 = ∅', len(val_p & set(ds[5000])) == 0))

# Вывод
failed = [name for name, ok in checks if not ok]
print(f'Проверок: {len(checks)}, пройдено: {len(checks) - len(failed)}')
if failed:
    for name in failed:
        print(f'  ✗ {name}')
else:
    print('Все проверки пройдены ✓')

In [ ]:
## Итог бейзлайна

import json

with open(f'{RESULTS_ROOT}/baseline.json') as f:
    b = json.load(f)

print('┌──────────────┬──────────┬───────────────────┬───────────┐')
print('│ Бенчмарк     │ Accuracy │ 95% CI            │ none_rate │')
print('├──────────────┼──────────┼───────────────────┼───────────┤')
for name, key in [('GSM8K test', 'gsm8k'), ('MATH-500', 'math500')]:
    acc = b[key]['accuracy']
    lo, hi = b[key]['ci']
    nr = b[key]['none_rate']
    print(f'│ {name:<12s} │ {acc:>7.1%}  │ [{lo:.3f}, {hi:.3f}]    │ {nr:>8.1%}  │')
print('└──────────────┴──────────┴───────────────────┴───────────┘')
print()
print('Парсер GSM8K: lenient (последнее число)')
print('Парсер MATH-500: строгий (\\boxed{})')

In [ ]:
from datasets import load_dataset
ds_raw = load_dataset('open-r1/OpenR1-Math-220k', split='train', name='default')

changed = 0
for ex in ds_raw:
    math_flags = ex.get('correctness_math_verify') or []
    llama_flags = ex.get('correctness_llama') or []
    old_idx, new_idx = None, None
    for i, gen in enumerate(ex['generations']):
        if old_idx is None and ((i < len(math_flags) and math_flags[i]) or (i < len(llama_flags) and llama_flags[i])):
            old_idx = i
        if new_idx is None and i < len(math_flags) and math_flags[i]:
            new_idx = i
    if old_idx != new_idx and new_idx is not None:
        changed += 1

print(f'Затронуто: {changed} из {len(ds_raw)}')